### Хэш-таблицы

Питоновский хэш от числа - это само число. Для строки это не так, и при изменении одного символа значение меняется очень сильно

In [3]:
# Хэширование строки
hash_value = hash('Hello, World!a')
print(hash_value)
hash_value = hash('Hello, World!b')
print(hash_value)

# Хэширование числа
hash_value = hash(12344)
print(hash_value)
hash_value = hash(12345)
print(hash_value)

-4879727982180751692
236434501945364896
12344
12345


### Работа со словарями

Сравним время работы словаря при обращении по двум разным типам ключей - числовым и строковым

In [7]:
import time

d = {}

start_time = time.time()

for j in range(100000):
    for i in range(10):
        d[i] = i + j

print(time.time() - start_time)

print(len(d))

0.052912235260009766
10


In [8]:
import time

d = {}

start_time = time.time()

for j in range(100000):
    for i in range(10):
        d[str(i)] = i + j

print(time.time() - start_time)

print(len(d))

0.08812189102172852
10


In [9]:
s = "aaaaaab"
    
strings = {}

for i in range(len(s) - 1):
    new_key = s[0 : i] + s[i + 2 :]
    
    print(new_key)
    
    if (new_key not in strings):
        strings.update({hash(new_key) : 1})

print(strings)
print(len(strings))

aaaab
aaaab
aaaab
aaaab
aaaab
aaaaa
{8780768563703691287: 1, 3977395868512878658: 1}
2


In [10]:
s = "abacabadabacaba"
    
strings = {}

for i in range(0, len(s) - 2, 2):
    new_key = s[i : i + 3]
    
    print(new_key)
    
    if (new_key not in strings):
        strings.update({hash(new_key) : 1})

print(strings)
print(len(strings))

aba
aca
aba
ada
aba
aca
aba
{-4719769938927825394: 1, -8459448819081703366: 1, 6945902693104215605: 1}
3


### Деревья отрезков (задача для домашнего выполнения)

- разберитесь в реализации дерева отрезков, приведенной ниже
- переделайте ее для нахождения среднего значения на непрерывном подотрезке

In [11]:
INFTY = float('inf')

class Stnode:
    def __init__(self, ind, val):
        self.ind = ind
        self.val = val
        self.leftmost = None
        self.rightmost = None

class Segment_tree:
    def __init__(self, arr):
        self.arr = arr
        
        self.h = 0
        self.datalen = len(self.arr)
        self.power = 1
        
        while (self.power < self.datalen):
            self.h += 1
            self.power *= 2
                
        print(f"elems {self.datalen}, power {self.power}, height {self.h}")
        
        # prepare array
        self.total_length = 2**(self.h + 1) - 1
        self.nodes = [None] * self.total_length
        self.first_data_position = self.total_length - self.power

        # copy data
        for i, el in enumerate(self.arr):
            self.nodes[self.first_data_position + i] = Stnode(self.first_data_position + i, el)
            self.nodes[self.first_data_position + i].leftmost = i
            self.nodes[self.first_data_position + i].rightmost = i
        
        for i in range(self.first_data_position + self.datalen, self.total_length):
            self.nodes[i] = Stnode(i, INFTY)
            self.nodes[i].leftmost = i
            self.nodes[i].rightmost = i
        
        # calc upwards
        for i in range(self.first_data_position - 1, -1, -1):
            left  = self.nodes[self.left(i)]
            right = self.nodes[self.right(i)]
            
            new_node = Stnode(i, min(left.val, right.val))
            new_node.leftmost  = left.leftmost
            new_node.rightmost = left.rightmost
            
            self.nodes[i] = new_node
    
    def left(self, i):
        ind = 2 * i + 1
        return ind if ind < len(self.nodes) else None
    
    def right(self, i):
        ind = 2 * i + 2
        return ind if ind < len(self.nodes) else None
    
    def parent(self, i):
        ind = (i - 1) // 2
        
        return ind if ind > 0 else None
    
    def find_min(self, l, r):
        indl, indr = l + self.first_data_position, r + self.first_data_position
        minl, minr = self.nodes[indl].val, self.nodes[indr].val
        
        node = self.nodes[indl]
        parent = self.nodes[self.parent(node.ind)]
        
        while(parent is not None and node.rightmost <= r):
            if (self.right(parent.ind) != node.ind):
                minl = min(minl, self.nodes[self.right(parent.ind)].val)
            
            node = parent
            parent = self.nodes[self.parent(node.ind)] if self.parent(node.ind) else None

        node = self.nodes[indr]
        parent = self.nodes[self.parent(node.ind)]
        
        while(parent is not None and node.leftmost >= l):
            if (self.left(parent.ind) != node.ind):
                minr = min(minr, self.nodes[self.left(parent.ind)].val)
            
            node = parent
            parent = self.nodes[self.parent(node.ind)] if self.parent(node.ind) else None
        
        return min(minl, minr)
    
stree = Segment_tree([1, 5, 3, 2, 4, 6])

print(stree.find_min(5, 6))

elems 6, power 8, height 3
4
